<img src="https://raw.githubusercontent.com/mctools/ncrystal-logo/main/png/logo-compact/ncrystal-logo-compact-64h.png" alt="NCrystal" width="158">

# Example: Water / Ice

In [ ]:
#Install software on Google Colab:
%pip -q install "ncrystal[cif,plot]" ipympl pandas ncrystal-plugin-WaterData
#Enable widgets on Google Colab:
from google.colab import output as _google_colab_output
_google_colab_output.enable_custom_widget_manager()
#Matplotlib plots:
%matplotlib inline
import matplotlib
matplotlib.rcParams.update({"figure.autolayout": True})
#Avoid warnings later on Google Colab (cf. mctools/ncrystal#192):
import gemmi
import spglib
#Import NCrystal:
import NCrystal as NC
assert NC.version_num >= 4004006, "NCrystal version 4.4.6 or later is needed"
NC.test() #< quick test that the installation works
#Check that NCrystal finds the plugin:
assert 'WaterData' in [ p[0] for p in NC.browsePlugins() ], 'NCrystal plugin WaterData not found'

<a id="waterice-intro"></a>

## Introduction
In this example we use NCrystal to see the wavelength dependence of the total cross section of light water and ice at different temperatures. This could be applied for the determination of ice in neutron imaging experiments, as presented in [1].

[1] M. Siegwart, R. Woracek, J.I. Marquez Damian, A.S. Tremsin, V. Manzi-Orezzoli, M. Strobl, T.J. Schmidt, and P. Boillat. "Distinction between super-cooled water and ice with high duty cycle time-of-flight neutron imaging." Review of Scientific Instruments 90, no. 10 (2019).

https://doi.org/10.1063/1.5110288

In this example we will show how to use additional data files, provided by an NCrystal plugin (with data from the [ncrystal-extra](https://github.com/mctools/ncrystal-extra) repository on GitHub), and how to create a new material using available information.

<a id="waterice-waterdata"></a>

## Water data from the WaterData plugin

When it comes to liquids, NCrystal is currently limited to using scattering kernels computed with other tools (i.e. the scattering kernel for liquids is not yet computed on the fly from the vibrational density of states). These precomputed kernels are rather large, and for that reason only the room-temperature files are available in the standard data library, which is installed with NCrystal itself. Data files for more temperatures are provided by the WaterData plugin, one of the NCrystal plugins listed at https://github.com/mctools/ncrystal/wiki/CuratedPlugins. This notebook therefore needs that plugin, which is installed together with its other requirements (see the top of the notebook). Note that NCrystal looks for plugins when it is first imported, so plugins must be installed before `import NCrystal`.

Let us use the Python API to see which light water data files are available (on the command line, `nctool --browse` lists all available files as well):

In [ ]:
for fn in NC.browseFiles():
  if 'LiquidWater' in fn.name:
    print(fn.fullKey)

Only one of these files comes with NCrystal itself, in its standard data library (the `stdlib::` file, at room temperature), while the WaterData plugin provides files for many more temperatures (the `plugins::WaterData/` files). The `stdlib::` file is the same as the `T293.6K` data file from the WaterData plugin.

We will need a few other imports as well in the rest of the notebook:

In [ ]:
import NCrystal.cifutils as nccif
import NCrystal.plot as ncplot
import numpy as np
import matplotlib.pyplot as plt
import pprint
import os

<a id="waterice-water"></a>

## Looking at the light water

First prepare the a list of water data files we wish to investigate:

In [ ]:
all_water = sorted( x.fullKey for x in NC.browseFiles()
                    if x.fullKey.startswith('plugins::WaterData/LiquidWater') )
pprint.pprint(all_water)

Let us plot the cross sections:

In [ ]:
E = np.geomspace(1e-4, 10, 1000)

plt.figure()
for fn in all_water:
  mat = NC.load(fn)
  plt.loglog(E, mat.scatter.xsect(E), label=f'T = {mat.info.getTemperature()} K')
plt.legend()

plt.xlabel('Energy [eV]')
plt.ylabel('Micro scattering cross section [b/atom]')

Note that the `NCrystal WARNING: Discarding .. edges ...` warnings above are harmless, and are caused NCrystal noticing the invalid areas of the original ENDF kernels.

The current scattering kernels for light water come from the ENDF/B-VIII.0 nuclear data library [1] and were produced using the CAB Model for Water [2]. Models for $T < 373.15$ K are at 1 atm, and for $T > 373.15$ K are at saturation pressure.

[1] D.A. Brown, M.B. Chadwick, R. Capote, A.C. Kahler, et al. "ENDF/B-VIII. 0: the 8th major release of the nuclear reaction data library with CIELO-project cross sections, new standards and thermal scattering data". Nuclear Data Sheets, 148, pp.1-142 (2018).
https://doi.org/10.1016/j.nds.2018.02.001

[2] J.I. Marquez Damian,J. R. Granada, and D. C. Malaspina. "CAB models for water: A new evaluation of the thermal neutron scattering laws for light and heavy water in ENDF-6 format." Annals of Nuclear Energy 65, 280-289 (2014).

https://doi.org/10.1016/j.anucene.2013.11.014

<a id="waterice-ice"></a>

## Create scattering kernel for ice Ih

We will now create a ncmat file for hexagonal ice (ice Ih) which is the phase that forms when water freezes at atmospheric pressure. Additional information on hexagonal ice and the phases of water can be found [here](https://water.lsbu.ac.uk/water/hexagonal_ice.html).

We will use the structure from the Crystallography Open Database [1], entry [1011023](https://www.crystallography.net/cod/1011023.html) [2], and the vibrational density of states used in the ENDF/B-VIII.0 scattering kernel, computed by Holmes and Zerkle [3].

[1] A. Vaitkus, A. Merkys, T. Sander, M. Quirós,P.A. Thiessen, E.E Bolton, and S. Gražulis, A workflow for deriving chemical entities from crystallographic data and its application to the Crystallography Open Database.
Journal of Cheminformatics, 15(1), (2023).

https://doi.org/10.1186/s13321-023-00780-2.

[2] J. D. Bernal, and R.H. Fowler. "A theory of water and ionic solution, with particular reference to hydrogen and hydroxyl ions." J. Chem. Phys 1, no. 8, 515-548 (1933).

http://dx.doi.org/10.1063/1.1749327

[3] J.C. Holmes, and M.L. Zerkle. "Production of the thermal scattering law and neutron scattering cross sec-tions for ice from first principles." International Conference on Nuclear Criticality Safety (ICNC 2015), Charlotte, NC, USA.

We start by retrieving the structure from the COD:

In [ ]:
os.environ['NCRYSTAL_ONLINEDB_CACHEDIR'] = './ncrystal_onlinedb_filecache'
cifsrc = nccif.CIFSource('codid::1011023')
print( cifsrc.load_data() )

Note: this CIF file corresponds to Ice Ih(a), with space group 185. Other forms of hexagonal ice exist [1], and this might be taken into account depending on the application.

[1] D. Jovanović, D. Zagorac, J.C Schön, B. Milovanović, and J. Zagorac. A new theoretical model for hexagonal ice, Ih (d), from first principles investigations. Zeitschrift für Naturforschung B, 75(1-2), pp.125-128 (2020).

https://doi.org/10.1515/znb-2019-0164

Now we make a NCMATComposer object with this structure:

In [ ]:
c_ice = NC.NCMATComposer.from_cif(cifsrc)

In [ ]:
print(c_ice())

The file contains the structural information, and the atom positions are verified to satisfy the symmetries of space group 185. The dynamics is represented with a simple Debye model as a place holder. We will replace this with the VDOS from Holmes and Zerkle.

We include here the vibrational density of states for H and O in light water ice. Curves were extracted from the ENDF-6 files using the one-phonon approximation to $S(0, \beta)$.

In [ ]:
txt = """0.000000000000000000e+00 0.000000000000000000e+00
5.060000000000000478e-04 2.883849022022742514e-01
1.012000000000000096e-03 1.153400150352600795e+00
1.517999999999999818e-03 2.367283816279119435e+00
2.024000000000000191e-03 3.497420702987856345e+00
2.530000000000000131e-03 4.471637932618838462e+00
3.035999999999999636e-03 5.955900760772344427e+00
3.542000000000000443e-03 7.477049638617704552e+00
4.048000000000000383e-03 9.044700529519186460e+00
4.553999999999999888e-03 1.061793535225325869e+01
5.060000000000000261e-03 1.229680504070817904e+01
6.324999999999999893e-03 1.930291141220044082e+01
7.589999999999998657e-03 3.296922597614996420e+01
8.855000000000000024e-03 4.693781596267880474e+01
1.012000000000000052e-02 4.540233355781982993e+01
1.138500000000000102e-02 3.508185808897206925e+01
1.264999999999999979e-02 2.874293722773296977e+01
1.517999999999999731e-02 1.883445349789817413e+01
1.771000000000000005e-02 1.615279548671844267e+01
2.024000000000000105e-02 2.042328727463937810e+01
2.277000000000000204e-02 1.762401730263019672e+01
2.529999999999999957e-02 2.132437305413203177e+01
2.783000000000000057e-02 4.891415518248106764e+01
3.035999999999999463e-02 2.718480284985898265e+01
3.289000000000000257e-02 1.769293571959826039e+01
3.542000000000000010e-02 2.883600077206273582e+01
3.794999999999999762e-02 2.665634143397740985e+01
4.048000000000000209e-02 1.376993503599563740e+00
4.300999999999999962e-02 6.397501554953829734e-04
4.554000000000000409e-02 5.470202598765482906e-04
4.806999999999998774e-02 3.804773183981115291e-04
5.059999999999999915e-02 2.383857092015670767e-04
5.313000000000000361e-02 1.758517990417824545e-04
5.566000000000000114e-02 1.717089685747877467e-04
5.818999999999999173e-02 1.415270112322381912e-04
6.071999999999998926e-02 1.095964336926452448e-04
6.325000000000000067e-02 1.111204047217788702e-04
6.578000000000000513e-02 9.953940985723399107e-05
6.831000000000000960e-02 5.679128970111942251e-05
7.084000000000000019e-02 3.787195300709431160e-05
7.336999999999999078e-02 3.074183800276910819e-05
7.589999999999999525e-02 1.290483726912145613e-05
7.842999999999999972e-02 2.404256671109077572e-06
8.096000000000000418e-02 3.180656520044201280e-06
8.348999999999999477e-02 4.996211049163529315e-06
8.601999999999999924e-02 1.424312142998701847e-01
8.855000000000000371e-02 1.179069212950002721e-01
9.108000000000000818e-02 1.733523983422859738e+00
9.360999999999999877e-02 2.000652545589134690e-01
9.613999999999997548e-02 1.123567310110189764e-05
9.866999999999999382e-02 1.383531729769606952e-05
1.011999999999999983e-01 1.389788123119954122e-05
1.037299999999999889e-01 1.125030273486978982e-05
1.062600000000000072e-01 5.827428038377292507e-01
1.087899999999999978e-01 6.697617556814178252e-01
1.113200000000000023e-01 1.207992012950673683e-01
1.138499999999999929e-01 1.237232256243846695e+00
1.163799999999999835e-01 9.100830054566003691e-01
1.189100000000000018e-01 6.951482944174336431e-02
1.214399999999999785e-01 3.810279341099103156e-01
1.239700000000000246e-01 6.953796143304357136e-01
1.265000000000000013e-01 1.565614818337411895e-01
1.290299999999999780e-01 1.391316679304783044e-05
1.315600000000000103e-01 1.312911158370812504e-05
1.340899999999999870e-01 9.933414303646770165e-06
1.366200000000000192e-01 6.685480794892814425e-06
1.391499999999999959e-01 4.313001724371301917e-06
1.416800000000000004e-01 4.662612978667513958e-06
1.442100000000000048e-01 4.582151293411740351e-06
1.467399999999999816e-01 3.094481066643653039e-06
1.492700000000000138e-01 3.039583050433871674e-06
1.517999999999999905e-01 3.422207609763860240e-06
1.543299999999999950e-01 2.002487403497484508e-06
1.568599999999999994e-01 1.115120859565699518e-06
1.593900000000000039e-01 1.287260468348693081e-06
1.619200000000000084e-01 6.804864159194056056e-07
1.644499999999999851e-01 1.549399313010874672e-07
1.669799999999999895e-01 9.951980742859653566e-08
1.695100000000000218e-01 1.763364717068570011e-07
1.720399999999999985e-01 2.526926674059059784e-07
1.745700000000000029e-01 2.653545779986962713e-07
1.771000000000000074e-01 3.956920825445296357e-07
1.796299999999999564e-01 4.969901880629441291e-07
1.821600000000000164e-01 6.624545397111023244e-07
1.846899999999999931e-01 1.412213648589005498e-06
1.872199999999999975e-01 3.252382039901963612e-06
1.897500000000000020e-01 8.556777362162727531e-06
1.922799999999999510e-01 1.091304948751262604e-05
1.948099999999999832e-01 1.655658133658928513e-05
1.973399999999999876e-01 2.624594461258680789e-01
1.998699999999999921e-01 2.351734405131690409e+00
2.023999999999999966e-01 7.697396484118211868e-01
2.049300000000000010e-01 2.133010954766428213e+00
2.074599999999999778e-01 3.695847225478283358e-05
2.099900000000000100e-01 2.796484790179166105e-05
2.125200000000000145e-01 2.425890094341358298e-05
2.150499999999999912e-01 1.506203551302586365e-05
2.175799999999999956e-01 8.740964774247490481e-06
2.201099999999999723e-01 5.775682307953858107e-06
2.226400000000000046e-01 4.767281826926283901e-06
2.251700000000000090e-01 5.636106198252105759e-06
2.276999999999999857e-01 6.080048371389239213e-06
2.302299999999999625e-01 5.290341561723014911e-06
2.327599999999999669e-01 5.373793264880461348e-06
2.352900000000000547e-01 4.727092809017465486e-06
2.378200000000000036e-01 2.869153705688560368e-06
2.403500000000000081e-01 2.222585826228122224e-06
2.428799999999999570e-01 9.836559402607070511e-07
2.454099999999999893e-01 6.351314929736053016e-08
2.479400000000000492e-01 5.817970807342710417e-09
2.504700000000000260e-01 1.145821653755291608e-09
2.530000000000000027e-01 9.998909681526179643e-11
2.555299999999999794e-01 1.982055748070002594e-11
2.580599999999999561e-01 1.566090466064390920e-11
2.605899999999999883e-01 1.359487576735446604e-11
2.631200000000000205e-01 1.174460909179000246e-11
2.656499999999999972e-01 9.230286460490579797e-12
2.681799999999999740e-01 7.270853415506054937e-12
2.707099999999999507e-01 5.222110795326685112e-12
2.732400000000000384e-01 3.459490643070369751e-12
2.757700000000000151e-01 2.096592858737407922e-12
2.782999999999999918e-01 1.368964749832412694e-12
2.808299999999999685e-01 1.120726683667904986e-12
2.833600000000000008e-01 2.256478210009728612e-12
2.858900000000000330e-01 1.001609254391501693e-08
2.884200000000000097e-01 2.302567377625905195e-08
2.909499999999999864e-01 5.669422588874445631e-08
2.934799999999999631e-01 2.539697681063101615e-08
2.960099999999999953e-01 3.790043181599647916e-08
2.985400000000000276e-01 4.131082388545300182e-09
3.010700000000000043e-01 4.026325756573600128e-12
3.035999999999999810e-01 1.093686423637686752e-09
3.061300000000000132e-01 2.606604510321965438e-08
3.086599999999999899e-01 2.152000692900502930e-08
3.111900000000000222e-01 3.250838626588698357e-08
3.137199999999999989e-01 5.367726215733498919e-08
3.162499999999999756e-01 3.292264226534639968e-08
3.187800000000000078e-01 3.286768268953506345e-08
3.213099999999999845e-01 6.034439966070248636e-08
3.238400000000000167e-01 2.144978272595770059e-08
3.263699999999999934e-01 3.215698369888215916e-08
3.288999999999999702e-01 1.507262651610132251e-08
3.314300000000000024e-01 2.142297891437490852e-09
3.339599999999999791e-01 5.838559293673130758e-10
3.364900000000000113e-01 1.390331099357339761e-08
3.390200000000000435e-01 3.208543626500598941e-08
3.415500000000000758e-01 3.625845482551586057e-08
3.440799999999999970e-01 7.361237443950054561e-08
3.466099999999999737e-01 1.541047912532941108e-07
3.491400000000000059e-01 1.667941466447290322e-07
3.516699999999999826e-01 2.397889037848473844e-07
3.542000000000000148e-01 3.829507392768694000e-07
3.567299999999999915e-01 5.072164847727268389e-07
3.592599999999999127e-01 8.245945798144106789e-07
3.617899999999999450e-01 1.670412844900979344e-06
3.643200000000000327e-01 3.691370313476315131e-06
3.668500000000000649e-01 6.506252144166972970e-06
3.693799999999999861e-01 7.809700106698804848e-06
3.719099999999999628e-01 1.260873589665825372e-05
3.744399999999999951e-01 2.218314520758850161e+00
3.769700000000000273e-01 8.658621940991413357e-01
3.795000000000000040e-01 5.060104812958891030e-01
3.820299999999999807e-01 1.011586222384029732e+00
3.845599999999999019e-01 4.402282385142110011e-01
3.870899999999999341e-01 7.923930408835356820e-01
3.896199999999999664e-01 2.922899837113844601e+00
3.921499999999999986e-01 1.754793512247840193e+00
3.946799999999999753e-01 2.504592551543724177e+00
3.972099999999999520e-01 6.025089222374397751e-05
3.997399999999999842e-01 4.857034247989964158e-05
"""
# From ENDF-6 file

holmes_O = np.fromstring(txt, sep=' ')
holmes_O = holmes_O.reshape(-1, 2)

In [ ]:
txt = """0.000000000000000000e+00 0.000000000000000000e+00
5.060000000000000478e-04 9.481969976656070068e-03
1.012000000000000096e-03 3.792329830552217812e-02
1.517999999999999818e-03 7.961955151797726893e-02
2.024000000000000191e-03 1.237345864328847633e-01
2.530000000000000131e-03 1.679503118679502582e-01
3.035999999999999636e-03 2.225506107383489196e-01
3.542000000000000443e-03 2.777824267920364631e-01
4.048000000000000383e-03 3.442900550430591511e-01
4.553999999999999888e-03 4.220368509412710245e-01
5.060000000000000261e-03 5.084022688234719478e-01
6.324999999999999893e-03 7.845498523048110018e-01
7.589999999999998657e-03 1.019992111524830491e+00
8.855000000000000024e-03 2.680891553042469955e+00
1.012000000000000052e-02 3.224700950914748621e+00
1.138500000000000102e-02 3.145002202032655791e+00
1.264999999999999979e-02 3.018117368757014862e+00
1.517999999999999731e-02 8.326946901054069761e-01
1.771000000000000005e-02 2.584248488544749219e-06
2.024000000000000105e-02 1.330240078832432005e-01
2.277000000000000204e-02 3.023644538633007278e-02
2.529999999999999957e-02 5.078432659026605595e-02
2.783000000000000057e-02 2.075702789955073513e+00
3.035999999999999463e-02 2.163538827209200655e+00
3.289000000000000257e-02 1.684280621352337803e+00
3.542000000000000010e-02 1.695701587600832339e+00
3.794999999999999762e-02 1.534824082768804887e+00
4.048000000000000209e-02 2.647894102456824012e-06
4.300999999999999962e-02 2.106371582140060183e-06
4.554000000000000409e-02 1.673144000907608366e-06
4.806999999999998774e-02 1.186963346858009909e-06
5.059999999999999915e-02 6.423875299545301701e-07
5.313000000000000361e-02 2.230628182467319015e-07
5.566000000000000114e-02 4.162998284030596753e-07
5.818999999999999173e-02 1.327811189403610618e-06
6.071999999999998926e-02 2.549053670410639611e-06
6.325000000000000067e-02 3.436338809503022472e-06
6.578000000000000513e-02 7.749574520149781279e-03
6.831000000000000960e-02 2.437577158326608817e+00
7.084000000000000019e-02 1.376023786347420597e+01
7.336999999999999078e-02 1.545567829638565804e+01
7.589999999999999525e-02 1.100675052938928644e+01
7.842999999999999972e-02 8.336120968995230029e+00
8.096000000000000418e-02 1.826149941477366667e+01
8.348999999999999477e-02 1.754927962575797196e+01
8.601999999999999924e-02 1.753305323153719542e+01
8.855000000000000371e-02 6.395695189705371320e+00
9.108000000000000818e-02 3.873317821720363430e+00
9.360999999999999877e-02 2.841759487909877313e-01
9.613999999999997548e-02 1.626954004739922244e+00
9.866999999999999382e-02 6.797624790611360268e+00
1.011999999999999983e-01 8.673146341301155360e+00
1.037299999999999889e-01 8.182317817153007766e+00
1.062600000000000072e-01 7.805199920065450492e+00
1.087899999999999978e-01 1.075854913708755589e+01
1.113200000000000023e-01 8.681314306125829106e+00
1.138499999999999929e-01 4.982520001025021017e+00
1.163799999999999835e-01 1.459102490093386351e+01
1.189100000000000018e-01 8.040661096025974786e+00
1.214399999999999785e-01 5.112995198932107632e-01
1.239700000000000246e-01 9.288092517353964820e-06
1.265000000000000013e-01 8.738817650899756038e-06
1.290299999999999780e-01 7.349785857678417663e-06
1.315600000000000103e-01 4.182485846081669102e-06
1.340899999999999870e-01 2.294199940231162687e-06
1.366200000000000192e-01 2.295086539180408950e-06
1.391499999999999959e-01 2.857842899486232335e-06
1.416800000000000004e-01 4.036267787858071418e-06
1.442100000000000048e-01 5.928729136392631436e-06
1.467399999999999816e-01 6.798749268747522048e-06
1.492700000000000138e-01 6.956815298038475040e-06
1.517999999999999905e-01 8.375425527195941858e-06
1.543299999999999950e-01 1.085783155685455984e-05
1.568599999999999994e-01 1.169737211113859764e-05
1.593900000000000039e-01 1.090869361167568672e-05
1.619200000000000084e-01 9.582643653152600835e-06
1.644499999999999851e-01 8.990424480339014989e-06
1.669799999999999895e-01 8.741213104327005907e-06
1.695100000000000218e-01 8.776337842565829858e-06
1.720399999999999985e-01 8.470468893375351818e-06
1.745700000000000029e-01 7.551942419920310924e-06
1.771000000000000074e-01 7.499851618032043693e-06
1.796299999999999564e-01 8.556520890588030089e-06
1.821600000000000164e-01 1.034072097359442200e-05
1.846899999999999931e-01 1.190495720722029107e-05
1.872199999999999975e-01 1.434571737388416174e-05
1.897500000000000020e-01 1.641593849240026976e-05
1.922799999999999510e-01 1.535175825671932781e-05
1.948099999999999832e-01 1.463699825323508226e-05
1.973399999999999876e-01 1.734726992700556636e+00
1.998699999999999921e-01 1.433219661734396233e+01
2.023999999999999966e-01 6.834766691639901914e+00
2.049300000000000010e-01 1.523161550552477550e+01
2.074599999999999778e-01 1.831038393320852053e+00
2.099900000000000100e-01 1.455755152556440303e-05
2.125200000000000145e-01 1.370640772847309061e-05
2.150499999999999912e-01 1.147628075642785476e-05
2.175799999999999956e-01 8.159042168958719970e-06
2.201099999999999723e-01 4.762758855186392871e-06
2.226400000000000046e-01 3.077106815645686430e-06
2.251700000000000090e-01 3.182517652314402641e-06
2.276999999999999857e-01 4.228114661582391843e-06
2.302299999999999625e-01 3.250038340970875058e-06
2.327599999999999669e-01 3.687229404768699736e-06
2.352900000000000547e-01 3.569953367246022889e-06
2.378200000000000036e-01 1.604530805993977590e-06
2.403500000000000081e-01 1.082768821408706220e-06
2.428799999999999570e-01 4.787592482833531197e-07
2.454099999999999893e-01 3.713342544682700450e-08
2.479400000000000492e-01 6.711614234674819888e-12
2.504700000000000260e-01 6.060495497547524698e-12
2.530000000000000027e-01 6.272336426206002136e-12
2.555299999999999794e-01 1.084903557225263254e-11
2.580599999999999561e-01 1.937623654960596382e-11
2.605899999999999883e-01 1.933335514705731704e-11
2.631200000000000205e-01 2.861936750187464373e-11
2.656499999999999972e-01 1.202573016003417973e-07
2.681799999999999740e-01 1.818465398435689700e-06
2.707099999999999507e-01 4.111359548551633110e-06
2.732400000000000384e-01 5.990409659070968502e-06
2.757700000000000151e-01 6.506896092849084591e-06
2.782999999999999918e-01 7.554974470464750476e-06
2.808299999999999685e-01 7.160220923871623427e-06
2.833600000000000008e-01 8.620224767091356547e-06
2.858900000000000330e-01 7.893694282548228231e-06
2.884200000000000097e-01 6.513675463430699767e-06
2.909499999999999864e-01 4.297025304201457466e-06
2.934799999999999631e-01 1.729050399967154560e-06
2.960099999999999953e-01 1.508838829737454615e-06
2.985400000000000276e-01 1.913827045117637160e-06
3.010700000000000043e-01 2.790915234027050661e-06
3.035999999999999810e-01 3.432332856671020680e-06
3.061300000000000132e-01 4.082693646682878733e-06
3.086599999999999899e-01 3.928035236805780580e-06
3.111900000000000222e-01 3.731060872505539431e-06
3.137199999999999989e-01 3.752786493918641020e-06
3.162499999999999756e-01 4.153478962605815586e-06
3.187800000000000078e-01 2.944349593166625916e-06
3.213099999999999845e-01 2.516762578800805462e-06
3.238400000000000167e-01 1.302640822803214332e-06
3.263699999999999934e-01 1.921996711061703504e-07
3.288999999999999702e-01 5.479377869614919691e-10
3.314300000000000024e-01 3.034533452124653357e-12
3.339599999999999791e-01 2.267331400897250357e-10
3.364900000000000113e-01 1.226053914492373576e-08
3.390200000000000435e-01 2.745810733710784462e-08
3.415500000000000758e-01 3.577212100979928429e-08
3.440799999999999970e-01 7.992713873942166892e-08
3.466099999999999737e-01 1.246021082050334182e-07
3.491400000000000059e-01 7.516685817139240700e-08
3.516699999999999826e-01 9.826494641470411766e-08
3.542000000000000148e-01 1.094957998727919359e-07
3.567299999999999915e-01 1.097907810879000962e-07
3.592599999999999127e-01 4.376212467403024985e-07
3.617899999999999450e-01 1.704527977000741893e-06
3.643200000000000327e-01 3.624286750815060180e-06
3.668500000000000649e-01 4.375186549511060154e-06
3.693799999999999861e-01 5.204540767692891657e-06
3.719099999999999628e-01 7.641643348358056857e-06
3.744399999999999951e-01 3.110286960572949155e+01
3.769700000000000273e-01 1.293772850287891885e+01
3.795000000000000040e-01 1.135918602299986802e+01
3.820299999999999807e-01 1.342973612070236555e+01
3.845599999999999019e-01 8.531778058395193653e+00
3.870899999999999341e-01 8.150043340931645730e+00
3.896199999999999664e-01 1.924229717856492172e+01
3.921499999999999986e-01 1.240923108182910362e+01
3.946799999999999753e-01 2.417049352844847476e+01
3.972099999999999520e-01 2.179279473764565153e-05
3.997399999999999842e-01 2.004821959992991026e-05
"""

# From ENDF-6 file

holmes_H = np.fromstring(txt, sep=' ')
holmes_H = holmes_H.reshape(-1, 2)

In [ ]:
plt.figure()
plt.plot(holmes_H[:,0], holmes_H[:,1], label='H')
plt.plot(holmes_O[:,0], holmes_O[:,1], label='O')
plt.legend()
plt.ylabel('VDOS [ev^-1]')
plt.xlabel('Energy [eV]')

We will use the NCrystal Phonon DOS Analyser to study the effect of the cutoff energy in the spectrum. Below this cutoff energy NCrystal will extrapolate the spectrum as a parabola.

In [ ]:
dosana = NC.PhononDOSAnalyser([('H', holmes_H[:,0], holmes_H[:,1]), ('O', holmes_O[:,0], holmes_O[:,1])])

In [ ]:
cutoff_candidates = np.linspace(0.002,0.01,5)
dosana.plot_cutoff_effects(cutoff_candidates,'H')
#Zoom of the same plot:
dosana.plot_cutoff_effects(cutoff_candidates,'H',xmin=0.00,xmax=0.035,ymin=0,ymax=10)

In [ ]:
cutoff_candidates = np.linspace(0.002,0.01,5)
dosana.plot_cutoff_effects(cutoff_candidates,'O')
#Zoom of the same plot:
dosana.plot_cutoff_effects(cutoff_candidates,'O',xmin=0.00,xmax=0.035,ymin=0,ymax=50)

$E_{cutoff} = 4$ meV is a reasonable value for the cutoff in both spectra.

In [ ]:
E_cutoff = 0.004
dosana_final = dosana.apply_cutoff(E_cutoff)

In [ ]:
dosana_final.plot()

Now we apply these spectra to the NCMATComposer object to get the full model. We add a comment to document the source of the dynamical data.

In [ ]:
dosana_final.apply_to( c_ice )
txt = """ ############################################

Vibrational density of states obtained from the ENDF/B-VIII.0 library
as computed by Holmes and Zerkle.

D.A. Brown, M.B. Chadwick, R. Capote, A.C. Kahler, et al.
"ENDF/B-VIII. 0: the 8th major release of the nuclear reaction data library
with CIELO-project cross sections, new standards and thermal scattering data".
Nuclear Data Sheets, 148, pp.1-142 (2018).
https://doi.org/10.1016/j.nds.2018.02.001

J.C. Holmes, and M.L. Zerkle. "Production of the thermal scattering law and
neutron scattering cross sec-tions for ice from first principles."
International Conference on Nuclear Criticality Safety (ICNC 2015),
Charlotte, NC, USA.

"""

c_ice.add_comments(txt)

print(c_ice())

Now that we have a complete model for ice, we can plot the cross sections to see the different components:

In [ ]:
ncplot.plot_xsect(c_ice, mode='ekin', xmin=1e-5, xmax=5)

and generate a ncmat file to use in our simulations:

In [ ]:
a = c_ice.write('ice.ncmat')

We will now compare this file with experimental data from the EXFOR database, [entry 31588003](http://www-nds.iaea.org/EXFOR/31588.003), as measured by Torres [1]. Cross sections from NCrystal are multiplied by 3 to obtain the value per molecule.

[1] L. Torres, J. R. Granada, and J. J. Blostein. "Total cross sections of benzene at 90 K and light water ice at 115 K." Nuclear Instruments and Methods in Physics Research Section B: Beam Interactions with Materials and Atoms 251, no. 1, 304-305 (2006).

https://doi.org/10.1016/j.nimb.2006.05.027

In [ ]:
x4_entries = [{'id':31588003, 'label':'Light water ice (115 K)', 'natom':3, 'data_csv':''',DatasetID,year1,author1,DATA (B) 0.1,ERR-S (B) 0.944,EN (EV) 1.1,TEMP (K) 2.1,TEMP-ERR (K) 2.911,zaTarg1,Targ1,Proj,Emission,Prod,MF,MT,ReacType,Quant1,nx,indVars,Reacode
0,31588003,2006,L.Torres+,45.6028,0.32854,89.1251,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
1,31588003,2006,L.Torres+,45.3728,0.35242,70.7946,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
2,31588003,2006,L.Torres+,45.6211,0.32784,56.2341,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
3,31588003,2006,L.Torres+,45.1527,0.31849,44.6684,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
4,31588003,2006,L.Torres+,45.0204,0.31241,35.4813,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
5,31588003,2006,L.Torres+,45.3172,0.32025,28.1838,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
6,31588003,2006,L.Torres+,45.3662,0.31707,22.3872,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
7,31588003,2006,L.Torres+,45.107,0.27999,17.7828,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
8,31588003,2006,L.Torres+,45.3289,0.28813,14.1254,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
9,31588003,2006,L.Torres+,45.4755,0.2923,11.2202,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
10,31588003,2006,L.Torres+,45.378,0.28086,8.91251,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
11,31588003,2006,L.Torres+,45.4481,0.27026,7.07946,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
12,31588003,2006,L.Torres+,45.7589,0.28868,5.62341,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
13,31588003,2006,L.Torres+,45.764,0.27639,4.46684,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
14,31588003,2006,L.Torres+,46.0125,0.272,3.54813,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
15,31588003,2006,L.Torres+,45.8607,0.26741,2.81838,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
16,31588003,2006,L.Torres+,46.1347,0.26619,2.23872,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
17,31588003,2006,L.Torres+,46.3887,0.28736,2.11349,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
18,31588003,2006,L.Torres+,46.5355,0.288,1.88365,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
19,31588003,2006,L.Torres+,46.7356,0.29008,1.6788,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
20,31588003,2006,L.Torres+,47.0037,0.28504,1.49624,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
21,31588003,2006,L.Torres+,47.2181,0.29187,1.33352,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
22,31588003,2006,L.Torres+,47.1149,0.28992,1.1885,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
23,31588003,2006,L.Torres+,47.4542,0.2911,1.05925,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
24,31588003,2006,L.Torres+,47.9418,0.2829,0.94406,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
25,31588003,2006,L.Torres+,48.0685,0.28976,0.8414,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
26,31588003,2006,L.Torres+,48.4278,0.28617,0.74989,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
27,31588003,2006,L.Torres+,48.7841,0.28144,0.66834,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
28,31588003,2006,L.Torres+,49.6171,0.28653,0.59566,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
29,31588003,2006,L.Torres+,50.3229,0.29743,0.53088,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
30,31588003,2006,L.Torres+,50.4928,0.29158,0.47315,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
31,31588003,2006,L.Torres+,50.735,0.29568,0.4217,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
32,31588003,2006,L.Torres+,51.4023,0.29252,0.37584,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
33,31588003,2006,L.Torres+,53.1813,0.29728,0.33497,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
34,31588003,2006,L.Torres+,54.4938,0.30917,0.29854,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
35,31588003,2006,L.Torres+,55.9213,0.3155,0.26607,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
36,31588003,2006,L.Torres+,57.5277,0.31846,0.23714,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
37,31588003,2006,L.Torres+,58.7882,0.32363,0.21135,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
38,31588003,2006,L.Torres+,59.7845,0.3376,0.19387,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
39,31588003,2006,L.Torres+,60.7767,0.3421,0.18302,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
40,31588003,2006,L.Torres+,61.7197,0.34471,0.17278,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
41,31588003,2006,L.Torres+,62.5904,0.34422,0.16312,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
42,31588003,2006,L.Torres+,63.7363,0.3455,0.15399,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
43,31588003,2006,L.Torres+,64.6691,0.35457,0.14538,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
44,31588003,2006,L.Torres+,65.7362,0.35839,0.13725,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
45,31588003,2006,L.Torres+,66.8783,0.36156,0.12957,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
46,31588003,2006,L.Torres+,67.5068,0.36849,0.12232,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
47,31588003,2006,L.Torres+,68.3472,0.36795,0.11548,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
48,31588003,2006,L.Torres+,68.9581,0.37315,0.10902,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
49,31588003,2006,L.Torres+,69.6299,0.37697,0.10292,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
50,31588003,2006,L.Torres+,70.0468,0.37778,0.09716,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
51,31588003,2006,L.Torres+,70.6475,0.37899,0.09173,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
52,31588003,2006,L.Torres+,71.2404,0.38608,0.0866,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
53,31588003,2006,L.Torres+,71.9682,0.38811,0.08175,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
54,31588003,2006,L.Torres+,72.7581,0.39144,0.07718,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
55,31588003,2006,L.Torres+,73.6991,0.39281,0.07286,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
56,31588003,2006,L.Torres+,74.8549,0.40295,0.06879,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
57,31588003,2006,L.Torres+,76.2901,0.411,0.06494,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
58,31588003,2006,L.Torres+,78.4252,0.41993,0.06131,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
59,31588003,2006,L.Torres+,80.5945,0.43263,0.05788,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
60,31588003,2006,L.Torres+,82.9146,0.44453,0.05464,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
61,31588003,2006,L.Torres+,85.189,0.45537,0.05158,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
62,31588003,2006,L.Torres+,87.5234,0.46494,0.0487,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
63,31588003,2006,L.Torres+,89.7208,0.47665,0.04597,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
64,31588003,2006,L.Torres+,91.9569,0.48844,0.0434,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
65,31588003,2006,L.Torres+,94.0594,0.49779,0.04097,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
66,31588003,2006,L.Torres+,96.6715,0.51071,0.03868,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
67,31588003,2006,L.Torres+,98.8095,0.5219,0.03652,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
68,31588003,2006,L.Torres+,101.052,0.53178,0.03447,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
69,31588003,2006,L.Torres+,103.4,0.54386,0.03255,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
70,31588003,2006,L.Torres+,105.52,0.55477,0.03073,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
71,31588003,2006,L.Torres+,107.726,0.56738,0.02901,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
72,31588003,2006,L.Torres+,109.895,0.57632,0.02738,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
73,31588003,2006,L.Torres+,111.741,0.5871,0.02585,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
74,31588003,2006,L.Torres+,113.345,0.59416,0.02441,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
75,31588003,2006,L.Torres+,115.164,0.60638,0.02304,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
76,31588003,2006,L.Torres+,117.124,0.61452,0.02175,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
77,31588003,2006,L.Torres+,118.995,0.62601,0.02054,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
78,31588003,2006,L.Torres+,120.77,0.63526,0.01939,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
79,31588003,2006,L.Torres+,123.004,0.64792,0.0183,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
80,31588003,2006,L.Torres+,124.742,0.65264,0.01728,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
81,31588003,2006,L.Torres+,126.695,0.66626,0.01631,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
82,31588003,2006,L.Torres+,128.801,0.67758,0.0154,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
83,31588003,2006,L.Torres+,129.982,0.68798,0.01454,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
84,31588003,2006,L.Torres+,131.191,0.69195,0.01372,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
85,31588003,2006,L.Torres+,131.826,0.69592,0.01296,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
86,31588003,2006,L.Torres+,133.068,0.70476,0.01223,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
87,31588003,2006,L.Torres+,134.354,0.71438,0.01155,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
88,31588003,2006,L.Torres+,135.651,0.71913,0.0109,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
89,31588003,2006,L.Torres+,136.562,0.72226,0.01029,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
90,31588003,2006,L.Torres+,137.42,0.73174,0.00972,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
91,31588003,2006,L.Torres+,139.047,0.75035,0.00917,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
92,31588003,2006,L.Torres+,139.866,0.75567,0.00866,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
93,31588003,2006,L.Torres+,141.116,0.76235,0.00818,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
94,31588003,2006,L.Torres+,142.226,0.76336,0.00772,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
95,31588003,2006,L.Torres+,142.887,0.77815,0.00729,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
96,31588003,2006,L.Torres+,143.813,0.78796,0.00688,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
97,31588003,2006,L.Torres+,145.12,0.79598,0.00649,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
98,31588003,2006,L.Torres+,146.507,0.81122,0.00613,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
99,31588003,2006,L.Torres+,147.018,0.81566,0.00579,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
100,31588003,2006,L.Torres+,148.459,0.84292,0.00546,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
101,31588003,2006,L.Torres+,148.415,0.83326,0.00516,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
102,31588003,2006,L.Torres+,148.42,0.85298,0.00487,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
103,31588003,2006,L.Torres+,150.079,0.89149,0.0046,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
104,31588003,2006,L.Torres+,151.189,0.8751,0.00434,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
105,31588003,2006,L.Torres+,151.769,0.87952,0.0041,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
106,31588003,2006,L.Torres+,153.074,0.92178,0.00387,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
107,31588003,2006,L.Torres+,153.619,0.9538,0.00365,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
108,31588003,2006,L.Torres+,154.578,0.94895,0.00345,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
109,31588003,2006,L.Torres+,155.049,0.99167,0.00325,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
110,31588003,2006,L.Torres+,155.983,1.02921,0.00307,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
111,31588003,2006,L.Torres+,155.238,0.92564,0.00299,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
112,31588003,2006,L.Torres+,157.512,0.94663,0.00266,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
113,31588003,2006,L.Torres+,160.018,1.02778,0.00237,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
114,31588003,2006,L.Torres+,160.449,1.07181,0.00211,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
115,31588003,2006,L.Torres+,162.206,1.13236,0.00188,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"
116,31588003,2006,L.Torres+,163.796,1.65057,0.00163,115,0.05,1001,1-H-WTR,N,TOT,,0,0,CST,CST,2,26,"1-H-WTR(N,TOT),,SIG/TMP"'''}]

In [ ]:
def plot_exfor(x4_entries, plot_wl=False):
    from urllib.request import urlopen, Request
    from urllib.error import HTTPError
    import pandas as pd
    import io
    for x4_entry in x4_entries:
        try:
            req = Request(
                url=f'https://nds.iaea.org/exfor/x4get?DatasetID={x4_entry["id"]}&op=csv',
                data=None,
                headers={
                    'User-Agent': 'Mozilla/5.0 (X11; Fedora; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0'
                }
            )
            with urlopen(req) as f:
                df = pd.read_csv(io.StringIO(f.read().decode('utf-8')))
        except HTTPError:
                df = pd.read_csv(io.StringIO(x4_entry['data_csv']))
        if 'EN (EV) 1.1' in list(df):
            x4_E, x4_xs = df['EN (EV) 1.1'].values, df['DATA (B) 0.1'].values
        elif 'EN (MEV) 1.1' in list(df):
            x4_E, x4_xs = 1e6*df['EN (MEV) 1.1'].values, df['DATA (B) 0.1'].values
        x = NC.ekin2wl(x4_E) if plot_wl else x4_E
        plt.plot(x, x4_xs/x4_entry['natom'], '.', label=x4_entry['label']+ f', X4: {x4_entry["id"]}')

In [ ]:
plt.figure()

plot_exfor(x4_entries)

mat = NC.load('ice.ncmat;temp=115K')
xs = mat.xsect(E)
plt.plot(E, xs, label = 'NCrystal model')

plt.legend(loc='upper right')
# plt.yscale('log')
plt.xscale('log')
plt.ylim(10, 70)
plt.xlim(1e-3, 5)
plt.xlabel("Energy [eV] ")
plt.ylabel("Cross-section (b per molecule)")

<a id="waterice-comparison"></a>

## Comparison of water and ice

Now that we have the models, we can compare the wavelength dependence of the total cross section of light water

In [ ]:
wl = np.linspace(2,9,1000)

mat_water350 = NC.load('plugins::WaterData/LiquidWaterH2O_T350.0K.ncmat')
mat_water293 = NC.load('plugins::WaterData/LiquidWaterH2O_T293.6K.ncmat')
mat_water283 = NC.load('plugins::WaterData/LiquidWaterH2O_T283.6K.ncmat')
mat_ice271 = NC.load('ice.ncmat;temp=271K')
mat_ice100 = NC.load('ice.ncmat;temp=100K')

#There are 3 atoms per water molecule:
atoms_per_molecule = 3

plt.figure()
plt.plot(wl, atoms_per_molecule*mat_water350.xsect(wl=wl), 'r', label='Water (77 C)')
plt.plot(wl, atoms_per_molecule*mat_water293.xsect(wl=wl), 'g', label='Water (22 C)')
plt.plot(wl, atoms_per_molecule*mat_water283.xsect(wl=wl), 'b', label='Water (10 C)')
plt.plot(wl, atoms_per_molecule*mat_ice271.xsect(wl=wl), 'k', label='Ice (-2 C)')
plt.plot(wl, atoms_per_molecule*mat_ice100.xsect(wl=wl), 'k--', label='Ice (-173 C)')
plt.legend()
plt.xlabel('Wavelength (Å)')
plt.ylabel('Micro total cross section [b/molecule]')
plt.show()